# Normalization
Min-max scaling on the wine data, then mean normalization, max-abs scaling and robust scaling on one small column.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler, RobustScaler, StandardScaler

# one look for every plot: Latin Modern font, white background
LOOK = dict(template="simple_white", font=dict(family="Latin Modern Roman", size=15))


def kde_table(long, by):
    """Density curve (KDE) of `value` for every group of the columns in `by`, as one long table."""
    parts = []
    for key, g in long.groupby(by, sort=False):
        xs = np.linspace(g["value"].min(), g["value"].max(), 200)
        part = pd.DataFrame({"value": xs, "density": gaussian_kde(g["value"])(xs)})
        parts.append(part.assign(**dict(zip(by, key))))
    return pd.concat(parts, ignore_index=True)

## Load the data
178 wines from three cultivars. We keep the class label and two inputs: alcohol and malic acid.

In [ ]:
# the file has no header row; keep only the first three columns
df = pd.read_csv("data/wine_data.csv", header=None, usecols=[0, 1, 2])
df.columns = ["Class label", "Alcohol", "Malic acid"]
df.head()

## Look at the data before scaling

In [ ]:
# density curve (KDE) of each input column
long = df.melt(id_vars="Class label", var_name="column", value_name="value")
fig = px.line(kde_table(long, ["column"]), x="value", y="density", facet_col="column", width=900, height=380)
fig.update_xaxes(matches=None).update_layout(**LOOK)

In [ ]:
# the two inputs against each other, coloured by class
px.scatter(df, x="Alcohol", y="Malic acid", color=df["Class label"].astype(str), labels={"color": "class"},
           width=600, height=450).update_layout(**LOOK)

## Split first, then scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="Class label"),   # inputs: Alcohol, Malic acid
    df["Class label"],                # output
    test_size=0.3, random_state=0)
X_train.shape, X_test.shape

## MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
# fit: learn each column's min and max, from the training set only
scaler.fit(X_train)
# transform: apply (x - min) / (max - min) to both sets, with the training numbers
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# what fit learned: min, max and range (max - min) of each column
print("min:  ", scaler.data_min_)
print("max:  ", scaler.data_max_)
print("range:", scaler.data_range_)

In [ ]:
# transform returns a NumPy array; put the column names back for easier reading
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
X_train_scaled.head()

In [ ]:
# the first row by hand: (x - min) / (max - min) should match the table above
x = X_train.iloc[0].to_numpy()
print(x, (x - scaler.data_min_) / scaler.data_range_)

## Before and after: describe

In [ ]:
np.round(X_train.describe(), 2)

In [ ]:
np.round(X_train_scaled.describe(), 2)

In [ ]:
# check: on the training set every column runs exactly from 0 to 1
assert np.allclose(X_train_scaled.min(), 0) and np.allclose(X_train_scaled.max(), 1)
# the test set was scaled with the training min and max, so it can step slightly outside 0 to 1
X_test_scaled.agg(["min", "max"])

## Effect of scaling: scatter plots

In [ ]:
# both versions in one long table, so one plot can draw them side by side
cls = y_train.astype(str).to_numpy()
both = pd.concat([X_train.assign(stage="Before scaling", cls=cls),
                  X_train_scaled.assign(stage="After scaling", cls=cls)], ignore_index=True)
fig = px.scatter(both, x="Alcohol", y="Malic acid", color="cls", facet_col="stage", labels={"cls": "class"},
                 width=900, height=400)
fig.update_xaxes(matches=None).update_yaxes(matches=None, showticklabels=True).update_layout(**LOOK)

## Effect of scaling: density curves (KDE)

In [ ]:
long = both.drop(columns="cls").melt(id_vars="stage", var_name="feature", value_name="value")
curves = kde_table(long, ["stage", "feature"])
fig = px.line(curves, x="value", y="density", color="feature", facet_col="stage", width=900, height=400)
fig.update_xaxes(matches=None).update_yaxes(matches=None, showticklabels=True).update_layout(**LOOK)

In [ ]:
# each column alone: the shape of the curve stays the same, only the numbers on the x axis change
fig = px.line(curves, x="value", y="density", facet_col="stage", facet_row="feature", width=900, height=600)
fig.update_xaxes(matches=None, showticklabels=True).update_yaxes(matches=None, showticklabels=True).update_layout(**LOOK)

## The other techniques on one small column
Five weights in kg; 130 is far from the rest.

In [ ]:
w = pd.DataFrame({"weight": [32, 54, 60, 67, 130]})
out = w.copy()
# mean normalization: scikit-learn has no class for it, so we write the formula ourselves
out["mean norm"] = (w["weight"] - w["weight"].mean()) / (w["weight"].max() - w["weight"].min())
# the other three have scikit-learn classes
out["min-max"] = MinMaxScaler().fit_transform(w)
out["max-abs"] = MaxAbsScaler().fit_transform(w)
out["robust"] = RobustScaler().fit_transform(w)
out["standard"] = StandardScaler().fit_transform(w)
out.round(3)

In [ ]:
# robust scaling by hand: (x - median) / (75th percentile - 25th percentile)
q1, med, q3 = w["weight"].quantile([0.25, 0.5, 0.75])
print("median", med, "IQR", q3 - q1)
print(((w["weight"] - med) / (q3 - q1)).round(3).tolist())

In [ ]:
# max-abs keeps zeros at zero and keeps the sign: useful for sparse data
s = pd.DataFrame({"x": [0, 0, -4, 0, 2, 0, 8]})
MaxAbsScaler().fit_transform(s).ravel()

## Try it yourself
Change the weights above (add a bigger outlier, or remove 130) and see which scalers move the other four values the most.